# 06 · Shared common: statistics, manifest and parity

This notebook exercises `common/stats.py` (per-column statistics, strict mode, `col_type` overrides), `common/manifest.py` (`unit_id`, `sample_plan`, `build_manifest`, `validate_manifest`) and the Left/Right parity guarantees, on **real data** from one of your pairs.
It mirrors `tests/unit/common/test_stats.py`, `tests/unit/common/test_manifest.py` and `tests/unit/common/test_parity.py`. Statistics are computed from one real day pulled from both sides; the manifest is built from the pair's real Stage 1 counts.
Nothing is written anywhere except a local work folder.

## How to approach

- **Prerequisites**: kernel = the repo `.venv` (`uv sync --extra dbx`, plus `--extra athena` / `--extra ldap` if the pair needs them). The repo `.env` holds the Left credentials (`<macro>_USR` / `_PWD` / `_DSN` or `ORACLE_*`). `~/.databrickscfg` has the `DBX_PROFILE` profile and `DTRACK_DBX_WAREHOUSE_ID` names a running SQL warehouse.
- **Run order**: top to bottom. Counting is quick; the one-day pull is the only slow cell (smallest common day, stops above `MAX_ROWS`). Sections 1-4 read the local CSVs; section 5 needs only the counts.
- **What to look at**: the side-by-side statistics table (which columns `match`, and why the others differ: `n_unique`, `hash_sum`, min/max), columns that only match under `strict`, the manifest windows (days with count differences are left out), and the parity checks.
- **Failure modes**: `CSV is missing columns` = a `col_map` name that is not in the pulled header; `row check has not passed` = no day with equal counts in the window (widen `DAYS` or pick another `TO_DATE`); `sampling requires a key` = the pair has `sample` but no `key`; a numeric column reported `categorical` on one side = text that is not a number on that side (or the known `col_map` case mismatch, see section 3).

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")      # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                               # one active pair in it
TO_DATE = None                                                                    # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                                          # window = DAYS days ending at TO_DATE
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")                        # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")                      # SQL warehouse id for the Right side
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")        # "inline" when presigned downloads are blocked
ENV_FILE = os.environ.get("DTRACK_ENV_FILE")                                      # None = <repo>/.env
MAX_ROWS = 200_000                                                                # refuse the one-day pull above this
KEY_COLUMNS = None                                                                # None = the pair's key; or ["ORDER_ID"]
GATE = "auto"                                                                     # row-check gate: auto | manual | ratio:0.1
WORKDIR = Path.home() / ".local/plans/nbs/_work/06"                               # local scratch folder
KEEP = False                                                                      # keep WORKDIR after the run

In [ ]:
# Imports used across the notebook
import json
import logging
import shutil
from datetime import date, timedelta
from types import SimpleNamespace

import pandas as pd
from dotenv import load_dotenv

import dtrack_left
from dtrack_left.backends.databricks import workspace
from dtrack_left.backends.sources import source_from_config
from dtrack_left.cli.config import load_config
from dtrack_local.pipeline import WarehouseTarget
from dtrack_local.warehouse import Warehouse

# The repo root (editable install: <repo>/src/dtrack_left/__init__.py) and its .env with the
# Left credentials (<macro>_USR / _PWD / _DSN, ORACLE_*, LDAP_DSN, DTRACK_ORACLE_MACROS ...)
REPO = Path(dtrack_left.__file__).resolve().parents[2]
env_file = Path(ENV_FILE) if ENV_FILE else REPO / ".env"
loaded = load_dotenv(env_file, override=False)
print("repo :", REPO)
print(".env :", env_file, "(loaded)" if loaded else "(not found - credentials must already be in the environment)")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

# The real pairs config, validated exactly like `dtrack row-check` does
config = load_config(CONFIG_PATH)
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
print("active pairs:", active)
if PAIR not in active:
    raise KeyError(f"PAIR={PAIR!r} is not an active pair in {CONFIG_PATH}")

# The pair, deep-copied, with its date range narrowed to the check window (TO_DATE back DAYS days)
to_date = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
from_date = (date.fromisoformat(to_date) - timedelta(days=DAYS - 1)).isoformat()
pair = json.loads(json.dumps(config["pairs"][PAIR]))
pair["fromDate"], pair["toDate"] = from_date, to_date
print(f"pair {PAIR}: window {from_date} .. {to_date}")
print("  left :", {k: v for k, v in pair["left"].items() if k not in ("user", "password", "dsn")})
print("  right:", pair["right"])

WORKDIR.mkdir(parents=True, exist_ok=True)

# Both copies of the shared common package, side by side
from dtrack_left.common import manifest as left_manifest, stats as left_stats
from dtrack_right.common import manifest as right_manifest, stats as right_stats
from dtrack_right.checks.col_check import compare_statistics

L = SimpleNamespace(stats=left_stats, manifest=left_manifest)
R = SimpleNamespace(stats=right_stats, manifest=right_manifest)

## Connect, count and pull one day

Preflight both sides, count rows per day for the window, then pull the smallest day present on both sides exactly as the col-check would (no sampling).

In [ ]:
# The real Left source (Oracle / Athena / file / duckdb) and the real Right side through the SQL warehouse
source = source_from_config(pair["left"])
warehouse = Warehouse(workspace(DBX_PROFILE), WAREHOUSE_ID, disposition=DISPOSITION)
right = WarehouseTarget(warehouse, pair["right"], WORKDIR / "right")

# Preflight both sides before any counting: Left SELECT ... WHERE 1=0, Right DESCRIBE QUERY
left_health = source.preflight()
right_health = right.preflight()
print("left  preflight:", left_health)
print("right preflight:", {k: v for k, v in right_health.items() if k != "columns"})
if not (left_health["ok"] and right_health["ok"]):
    raise RuntimeError("fix the preflight error above before going on")

In [ ]:
# Exact per-day counts on both sides for the window (the Stage 1 row-check numbers)
left_counts = source.partition_counts(from_date, to_date)
right_counts = right.partition_counts(from_date, to_date)
both = sorted(set(left_counts) & set(right_counts))
display(pd.DataFrame(
    [{"day": d, "left": left_counts.get(d), "right": right_counts.get(d)}
     for d in sorted(set(left_counts) | set(right_counts))]
))
if not both:
    raise RuntimeError("no day present on both sides: check date_col / date_type / where, or widen DAYS")

In [ ]:
# Column names: Left value columns (date_col excluded), keys, their Right names and key families
date_col = pair["left"]["date_col"]
value_cols = [c for c in pair["col_map"] if c.lower() != date_col.lower()]
keys = list(KEY_COLUMNS or [k.strip() for k in str(pair.get("key") or "").split(",") if k.strip()])
if not keys:
    keys = value_cols[:1]
    print(f"pair has no key; using {keys} as the sampling key for this demo (set KEY_COLUMNS to choose)")
right_keys = [pair["col_map"].get(k, k) for k in keys]
families = [dict(pair.get("key_families") or {}).get(k, "string") for k in keys]
print("value columns:", value_cols)
print("keys:", keys, "-> right", right_keys, "families", families)

# The smallest day present on both sides, so the pull stays small (MAX_ROWS guards it)
DAY = min(both, key=lambda d: left_counts[d])
if left_counts[DAY] > MAX_ROWS:
    raise RuntimeError(f"{DAY} has {left_counts[DAY]} Left rows (> MAX_ROWS); raise MAX_ROWS or pick another pair")
print(f"pulling {DAY}: left={left_counts[DAY]} right={right_counts[DAY]} rows")

# Left: the same pull_window the col-check uses (date_col + keys + value columns, raw text CSV)
left_csv = source.pull_window([DAY], value_cols, WORKDIR / "left.csv", key_columns=keys)

# Right: the same window SQL WarehouseTarget builds (every column CAST AS STRING), streamed to CSV.
# window_sql only reads col_map / columns / right_key_cols / sample from the manifest; we pass just those.
mini_manifest = {"col_map": pair["col_map"], "columns": value_cols, "right_key_cols": right_keys, "sample": {"mode": "all"}}
right_sql, right_selected = right.window_sql(mini_manifest, {"partitions": [DAY]})
print("right SQL:", right_sql)
right_rows = warehouse.to_csv(right_sql, WORKDIR / "right.csv", right_selected)
right_csv = WORKDIR / "right.csv"

left_frame = pd.read_csv(left_csv, dtype=object)
right_frame = pd.read_csv(right_csv, dtype=object)
print(f"left.csv  {len(left_frame)} rows  {list(left_frame.columns)}")
print(f"right.csv {len(right_frame)} rows  {list(right_frame.columns)}")
display(left_frame.head(3))
display(right_frame.head(3))

## 1 · column_statistics side by side

`compute_csv_statistics` reads the CSV as text and returns, per column: `col_type` (numeric when every non-null value parses), counts, `hash_sum` (sum of SHA-256/64 of each value), mean/std or length stats, min/max and top-10. The Right side uses the `col_map` names; `compare_statistics` (the Right job's comparator) gives the verdict.

In [ ]:
# The pair's own settings drive the statistics, exactly as in the manifest
strict = bool(pair.get("strict"))
overrides = dict(pair.get("col_type_overrides") or {})
col_map = pair["col_map"]
print("strict:", strict, "| col_type_overrides:", overrides or "(none)")

left_result = L.stats.compute_csv_statistics(left_csv, value_cols, strict=strict, col_types=overrides)
right_result = R.stats.compute_csv_statistics(right_csv, [col_map[c] for c in value_cols], strict=strict,
                                              col_types={col_map[k]: v for k, v in overrides.items()})

# One row per column: the main fields from both sides and the comparator's verdict
rows = []
for column in value_cols:
    a, b = left_result[column], right_result[col_map[column]]
    verdict = compare_statistics(pair=PAIR, window=DAY, column=column, left=a, right=b)
    rows.append({
        "column": column, "right": col_map[column], "type L/R": f"{a['col_type']}/{b['col_type']}",
        "n_total": f"{a['n_total']}/{b['n_total']}", "n_missing": f"{a['n_missing']}/{b['n_missing']}",
        "n_unique": f"{a['n_unique']}/{b['n_unique']}", "hash_sum": a["hash_sum"] == b["hash_sum"],
        "min": f"{a['min_val']} / {b['min_val']}", "max": f"{a['max_val']} / {b['max_val']}",
        "verdict": verdict["verdict"],
    })
stats_table = pd.DataFrame(rows)
display(stats_table)

In [ ]:
# Full detail for the first column that differs (or the first column when all match)
target = next((r["column"] for r in rows if r["verdict"] == "diff"), value_cols[0])
detail = pd.DataFrame({"left": left_result[target], "right": right_result[col_map[target]]})
print("column:", target)
display(detail)

## 2 · Canonical decimal text and strict mode

For numeric columns, values are hashed after `collapse_decimal`, so `7.50` and `7.5` hash the same. `strict=True` additionally lower-cases and collapses every value before typing, so text that differs only in case also matches. This compares `hash_sum` for each column with strict off and on.

In [ ]:
# hash_sum equality per column, loose vs strict, on the same real CSVs
loose_l = L.stats.compute_csv_statistics(left_csv, value_cols, strict=False)
loose_r = R.stats.compute_csv_statistics(right_csv, [col_map[c] for c in value_cols], strict=False)
strict_l = L.stats.compute_csv_statistics(left_csv, value_cols, strict=True)
strict_r = R.stats.compute_csv_statistics(right_csv, [col_map[c] for c in value_cols], strict=True)
display(pd.DataFrame([{
    "column": c,
    "type": loose_l[c]["col_type"],
    "hash_sum loose": loose_l[c]["hash_sum"] == loose_r[col_map[c]]["hash_sum"],
    "hash_sum strict": strict_l[c]["hash_sum"] == strict_r[col_map[c]]["hash_sum"],
    "n_unique loose L/R": f"{loose_l[c]['n_unique']}/{loose_r[col_map[c]]['n_unique']}",
    "n_unique strict L/R": f"{strict_l[c]['n_unique']}/{strict_r[col_map[c]]['n_unique']}",
} for c in value_cols]))

In [ ]:
# For numeric columns: raw text forms that differ between the sides but collapse to the same canonical text
from dtrack_left.common.hashing import collapse_decimal

for c in value_cols:
    if loose_l[c]["col_type"] != "numeric":
        continue
    raw_left = set(left_frame[c].dropna().str.strip())
    raw_right = set(right_frame[col_map[c]].dropna().str.strip())
    only = sorted(raw_left - raw_right)[:5]
    print(f"{c:25} left-only raw text {only} -> canonical {[collapse_decimal(v) for v in only]}"
          f" | canonical sets equal: {set(map(collapse_decimal, raw_left)) == set(map(collapse_decimal, raw_right))}")

# Edge cases taken from the real data: one value has no sample std
first_numeric = next((c for c in value_cols if loose_l[c]["col_type"] == "numeric"), None)
if first_numeric:
    print("single value std:", L.stats.column_statistics(left_frame[first_numeric].dropna().head(1))["std"])
# Illustrative literal: an all-null column is categorical with empty extremes
print("all-null column:", {k: v for k, v in L.stats.column_statistics(pd.Series([None, None], dtype=object)).items()
                           if k in ("col_type", "n_missing", "min_val", "length_max")})

## 3 · col_type overrides and column-name case

`col_type_overrides` lets the pair force `numeric` or `categorical` regardless of the values. Below, the first numeric column is forced categorical and the first categorical column forced numeric, to show what changes. Then the `col_map` Right names are checked against `DESCRIBE` for exact case (a known issue: a case mismatch silently turns a numeric column categorical).

In [ ]:
# Forcing the type on a real column (illustrative overrides, not written to the config)
first_text = next((c for c in value_cols if loose_l[c]["col_type"] == "categorical"), None)
for column, forced in ((first_numeric, "categorical"), (first_text, "numeric")):
    if column is None:
        continue
    before = L.stats.column_statistics(left_frame[column])
    after = L.stats.column_statistics(left_frame[column], col_type=forced)
    keys_shown = ["col_type", "n_unique", "mean", "std", "min_val", "max_val", "length_avg", "top_10"]
    print(f"--- {column}: inferred {before['col_type']} -> forced {forced}")
    display(pd.DataFrame({"inferred": {k: before[k] for k in keys_shown}, "forced": {k: after[k] for k in keys_shown}}))

# compute_csv_statistics refuses a column that is not in the CSV header
try:
    L.stats.compute_csv_statistics(left_csv, value_cols + ["NOT_A_COLUMN"])
except ValueError as exc:
    print("refused:", exc)

In [ ]:
# col_map Right names vs DESCRIBE on the warehouse: exact spelling required
described = set(right_health["columns"])
wanted = [pair["right"]["date_col"], *col_map.values()]
wrong = {w: next((a for a in described if a.lower() == w.lower()), "missing") for w in wanted if w not in described}
print("col_map right names that differ from DESCRIBE:", wrong or "none")

## 4 · compare_statistics verdicts

The Right job compares the Left unit payload with its own statistics. A unit is `match` only when the three counts are equal and mean/std/length/hash_sum/min/max all agree (floats within 1e-6). A perturbed copy shows a `diff`.

In [ ]:
# Verdict detail for every column: which checks failed
checks = ["n_total_diff", "n_missing_diff", "n_unique_diff", "mean_match", "std_match", "length_avg_match",
          "length_max_match", "hash_sum_match", "min_match", "max_match"]
verdicts = [compare_statistics(pair=PAIR, window=DAY, column=c, left=left_result[c], right=right_result[col_map[c]]) for c in value_cols]
display(pd.DataFrame([{"column": v["column"], **{k: v[k] for k in checks}, "verdict": v["verdict"]} for v in verdicts]))

# Same Left statistics on both sides of the comparator -> match; a nudged mean -> diff (illustrative)
sample_col = value_cols[0]
same = compare_statistics(pair=PAIR, window=DAY, column=sample_col, left=left_result[sample_col], right=left_result[sample_col])
nudged = {**left_result[sample_col], "mean": (left_result[sample_col]["mean"] or 0) + 1, "n_unique": left_result[sample_col]["n_unique"] + 1}
other = compare_statistics(pair=PAIR, window=DAY, column=sample_col, left=left_result[sample_col], right=nudged)
print("left vs itself:", same["verdict"], "| left vs nudged copy:", other["verdict"])

## 5 · The manifest from the real row check

The Right side reconciles the two count sets, applies the gate and, when it passes, builds the manifest: one window per vintage bucket of **matching** days (days with a count difference or present on one side only are left out), `expected_units = windows x columns`, and the sampling plan.

In [ ]:
from dtrack_left.checks.row_check import make_run_id, public_pair_config
from dtrack_right.checks.row_check import evaluate_gate, reconcile_partitions, summarize_partitions

# The same Stage 1 document the Right job writes, from the real counts
partitions = reconcile_partitions(left_counts, right_counts, pair.get("date_synonyms"))
row_check = {"partitions": partitions, "summary": summarize_partitions(partitions), "gate_decision": evaluate_gate(partitions, GATE)}
print("summary:", row_check["summary"])
print("gate   :", row_check["gate_decision"])
display(pd.DataFrame([{"day": d, **r} for d, r in partitions.items()]))

# The manifest, built from the credential-free pair config the Left side publishes
run_id = make_run_id({"pairs": {PAIR: pair}})
public = public_pair_config(pair)
try:
    manifest = L.manifest.build_manifest(run_id, PAIR, public, row_check)
except ValueError as exc:
    manifest = None
    print("no manifest:", exc)
if manifest:
    print("run_id:", run_id, "| vintage:", manifest["vintage"], "| sample:", manifest["sample"])
    print("columns:", manifest["columns"])
    print("keys:", manifest["key_cols"], "->", manifest["right_key_cols"], manifest["key_families"])
    display(pd.DataFrame(manifest["windows"]))
    print("expected_units:", manifest["expected_units"], "=", len(manifest["windows"]), "windows x", len(manifest["columns"]), "columns")
    L.manifest.validate_manifest(manifest)
    print("validate_manifest: ok")

In [ ]:
# unit_id for every real unit: stable, slugged, collision-resistant (a digest of the exact names)
if manifest:
    ids = [L.manifest.unit_id(PAIR, w["bucket"], c) for w in manifest["windows"] for c in manifest["columns"]]
    print(len(ids), "units,", len(set(ids)), "distinct; first:", ids[:3])
    print("stable:", ids == [L.manifest.unit_id(PAIR, w["bucket"], c) for w in manifest["windows"] for c in manifest["columns"]])
    print("same in the right copy:", ids == [R.manifest.unit_id(PAIR, w["bucket"], c) for w in manifest["windows"] for c in manifest["columns"]])

In [ ]:
# sample_plan: the pair's real value first, then illustrative literals (valid and refused)
print("pair sample", repr(pair.get("sample")), "->", L.manifest.sample_plan(pair.get("sample")))
for raw in [None, "", 20, "20", 1.0, "0.5", "1e-9", True, 1.5, 0.0, 0]:
    try:
        print(f"{raw!r:8} -> {L.manifest.sample_plan(raw)}")
    except (TypeError, ValueError) as exc:
        print(f"{raw!r:8} -> refused ({type(exc).__name__}: {exc})")

In [ ]:
import copy

# Deliberately broken copies of the real pair / row check / manifest: each must be refused (illustrative)
def variant(**changes):
    spec = copy.deepcopy(public)
    spec.update(changes)
    return spec

first_key = (manifest or {}).get("key_cols", keys)[:1] or keys[:1]
pair_cases = {
    "right not databricks": (variant(right={"source": "csv"}), row_check),
    "gate not passed": (public, {**row_check, "gate_decision": {"passed": False}}),
    "no value columns": (variant(col_map={pair["left"]["date_col"]: pair["right"]["date_col"]}), row_check),
    "bad key family": (variant(key_families={first_key[0]: "blob"}, key=first_key[0]), row_check),
    "sample without key": (variant(key="", sample=0.5), row_check),
    "override unknown column": (variant(col_type_overrides={"NOPE": "numeric"}), row_check),
}
for label, (spec, doc) in pair_cases.items():
    try:
        L.manifest.build_manifest(run_id, PAIR, spec, doc)
        print(f"{label:24} -> accepted")
    except ValueError as exc:
        print(f"{label:24} -> refused: {exc}")

if manifest:
    doc_cases = {
        "missing windows": lambda m: m.pop("windows"),
        "protocol version": lambda m: m.update(protocol_version=99),
        "expected_units": lambda m: m.update(expected_units=m["expected_units"] + 1),
        "unsorted partitions": lambda m: m["windows"][0].update(partitions=list(reversed(m["windows"][0]["partitions"])) * 2),
    }
    for label, change in doc_cases.items():
        broken = copy.deepcopy(manifest)
        change(broken)
        try:
            L.manifest.validate_manifest(broken)
            print(f"{label:24} -> accepted")
        except ValueError as exc:
            print(f"{label:24} -> refused: {exc}")

## 6 · Parity between the two packages

The Left package and the Databricks job ship separate copies of `common/`. They must stay byte-identical, produce identical statistics and manifests, and normalise keys the same way. The `disabled` decorator (used today on `HadoopSource`) is checked on both sides.

In [ ]:
# Byte-identical common/ in the two installed packages
left_dir, right_dir = Path(left_stats.__file__).parent, Path(right_stats.__file__).parent
names = sorted({p.name for p in left_dir.glob("*.py")} | {p.name for p in right_dir.glob("*.py")})
for name in names:
    a, b = left_dir / name, right_dir / name
    state = "identical" if a.exists() and b.exists() and a.read_bytes() == b.read_bytes() else "DRIFTED / missing"
    print(f"{name:15} {state}")

# Same real CSV through both copies -> identical statistics; same row check -> identical manifest
print("stats identical across copies:",
      L.stats.compute_csv_statistics(left_csv, value_cols) == R.stats.compute_csv_statistics(left_csv, value_cols))
if manifest:
    print("manifest identical across copies:", manifest == R.manifest.build_manifest(run_id, PAIR, public, row_check))

In [ ]:
from dtrack_left.common import hashing as lh
from dtrack_right.common import hashing as rh

# Key normalisation on real key values, both copies, every family (leading zeros must survive)
values = pd.concat([left_frame[k] for k in keys] + [right_frame[k] for k in right_keys]).dropna().drop_duplicates().head(2000)
def both(v, f):
    try:
        return lh.normalize_value(v, f), rh.normalize_value(v, f)
    except ValueError as exc:
        return ("err", "err")
mismatch = [(v, f) for v in values for f in ("string", "numeric", "date") if both(v, f)[0] != both(v, f)[1]]
lead = [v for v in values if str(v).strip().startswith("0") and len(str(v).strip()) > 1 and "." not in str(v)][:3]
print(f"{len(values)} real key values x 3 families: {len(mismatch)} mismatches")
print("leading-zero keys kept as-is:", [(v, lh.normalize_value(v, "numeric")) for v in lead] or "(none in this day)")

In [ ]:
import importlib
from dtrack_left.backends.sources import HadoopSource

# A real disabled class: HadoopSource keeps its attributes but refuses to instantiate
print("HadoopSource.dialect:", HadoopSource.dialect)
try:
    source_from_config({"source": "hadoop", "table": "t", "date_col": "dt"})
except NotImplementedError as exc:
    print("refused:", exc)

# The decorator itself, from both packages, on a throwaway function (illustrative)
for side in ("dtrack_left", "dtrack_right"):
    disabled = importlib.import_module(f"{side}.common").disabled

    @disabled("not fixed")
    def pull(table):
        return table

    try:
        pull("t")
    except NotImplementedError as exc:
        print(f"{side}: {exc} | name kept: {pull.__name__} | original still callable: {pull.__wrapped__('t')}")

## Cleanup

Only the local `WORKDIR` was written (`left.csv`, `right.csv`).

In [ ]:
# Remove the local working folder unless KEEP=True (nothing else was written anywhere)
if KEEP:
    print("kept", WORKDIR)
else:
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print("removed", WORKDIR)

## What to check

- Section 1: every column `match`, or a `diff` you can explain from the detail table (real drift, sampling, a type disagreement).
- Section 2: columns that match only with `strict` on are candidates for `"strict": true` in the pair.
- Section 3: no `col_map` Right name differs from `DESCRIBE`.
- Section 5: the gate passes, windows exclude every non-matching day, `expected_units = windows x columns`, every broken copy is refused.
- Section 6: all common files `identical`, identical stats / manifest across copies, 0 normalisation mismatches, `HadoopSource` refused.